# 3 Sep

# Parsing and checking out the coconut_with_cids.csv so I can ensure completeness for the DB and remove things I dont need like the descriptors and the organisms

## Getting an idea of whats here what is missing and what is distinct

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)

columns = coco.columns
columns.to_list()

entries = []
for col in columns:
    num_entries = len(coco[col])
    missing = coco[col].isnull().sum()
    distinct = coco[col].nunique()
    entries.append([col,num_entries,missing,distinct])

df_out = pd.DataFrame(entries,columns=['column_name','number_of_entries','missing_entries','distinct_entries'])
print(df_out)

                         column_name  number_of_entries  missing_entries  \
0                         identifier             738827                0   
1                   canonical_smiles             738827                0   
2                     standard_inchi             738827                0   
3                 standard_inchi_key             738827                0   
4                               name             738827           375404   
5                         iupac_name             738827            74593   
6                   annotation_level             738827                0   
7                   total_atom_count             738827                0   
8                   heavy_atom_count             738827                0   
9                   molecular_weight             738827                0   
10            exact_molecular_weight             738827                0   
11                 molecular_formula             738827                0   
12          

## Checking the coconut paper (old paper to be fair) claim that all compounds have a name or iupac

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)
count = 0
for idx, row in coco.iterrows():
    if pd.isna(row.name) and pd.isna(row.iupac_name) and pd.isna(row.synonyms):
        count +=   1

print(f'There are {count} entries with no obvious name')

There are 0 entries with no obvious name


## Giving each compound a name and dropping some of the columns I don't need

In [3]:
# im lazy just printing column names so I can copy paste
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)

cols = coco.columns
cols = cols.to_list()
print(cols)

['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name', 'annotation_level', 'total_atom_count', 'heavy_atom_count', 'molecular_weight', 'exact_molecular_weight', 'molecular_formula', 'alogp', 'topological_polar_surface_area', 'rotatable_bond_count', 'hydrogen_bond_acceptors', 'hydrogen_bond_donors', 'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski', 'lipinski_rule_of_five_violations', 'aromatic_rings_count', 'qed_drug_likeliness', 'formal_charge', 'fractioncsp3', 'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar', 'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework', 'np_likeness', 'chemical_class', 'chemical_sub_class', 'chemical_super_class', 'direct_parent_classification', 'np_classifier_pathway', 'np_classifier_superclass', 'np_classifier_class', 'np_classifier_is_glycoside', 'organisms', 'collections', 'dois', 'synonyms', 'cas', 'coconut_id', 'cid']


In [4]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', usecols=['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name',
                                                                                               'np_likeness','collections', 'dois', 'synonyms', 'cas', 'coconut_id', 'cid'], low_memory=False)
count = 0
for idx, row in coco.iterrows():
    if pd.isna(row.name) and pd.isna(row.iupac_name):
        count += 1
print(count)


0


## for the above I was going to sort out naming but apparently no entry is missing its iupac and name but it looks like the iupacs missinga are not necessarily all unresolveable so that is the job here but I am writing this so it can be done on wonko

- about 26% of the compounds that do not have an IUPAC do have a CID so I will just send the ones with cids to wonko to try
- first code block generates the file for wonko the second will be the actual script 

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', usecols=['identifier', 'iupac_name', 'cid'], low_memory=False)

df = coco[coco["iupac_name"].isna() | coco["iupac_name"].astype(str).str.strip().eq("")].copy()

df.to_csv('/home/school/masters/Scripts/coconut_full/coconut_compounds_need_iupac_for_wonko.csv',index=False)

In [ ]:
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out

in_path = '/nlustre/users/nathanc/coconut_compounds_need_iupac_for_wonko.csv'
results_path  = '/nlustre/users/nathanc/coconut_now_with_iupac_from_pubchem.csv'
error_path = '/nlustre/users/nathanc/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
err = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if pd.notna(row.cid):
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)

            if iupac:
                df.at[idx,'iupac_name'] = iupac
            else:
                err.append([cid,errorz])
        count += 1
        
        if count % 1000 == 0:
                df.to_csv(results_path,index=False)
                errors = pd.DataFrame(err, columns = ['cid','err'])
                errors.to_csv(error_path, index=False)
        print(progress_bar(count,length))

df.to_csv(results_path, index=False)
errors = pd.DataFrame(err, columns = ['cid','err'])
errors.to_csv(error_path, index=False)


# 4 Sep

## Wonko made a nice file and error file for the iupac getter stuff. (coconut_full/iupac_getter_errors.csv,coconut_full/coconut_now_with_iupac_from_pubchem.csv)
- just checking the error file to see whats up

In [ ]:
The below is accidental copy paste see labbook for output
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out

in_path = '/nlustre/users/nathanc/coconut_compounds_need_iupac_for_wonko.csv'
results_path  = '/nlustre/users/nathanc/coconut_now_with_iupac_from_pubchem.csv'
error_path = '/nlustre/users/nathanc/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
err = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if pd.notna(row.cid):
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)

            if iupac:
                df.at[idx,'iupac_name'] = iupac
            else:
                err.append([cid,errorz])
        count += 1
        
        if count % 1000 == 0:
                df.to_csv(results_path,index=False)
                errors = pd.DataFrame(err, columns = ['cid','err'])
                errors.to_csv(error_path, index=False)
        print(progress_bar(count,length))

df.to_csv(results_path, index=False)
errors = pd.DataFrame(err, columns = ['cid','err'])
errors.to_csv(error_path, index=False)




2537
                                            error_type  count
0                                             no iupac   2519
169                                 retry uncsucessful     15
215  <urlopen error [Errno -2] Name or service not ...      3


# 4/7 sep

## retrying the 18 that didn't come back as no iupac

In [8]:
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

in_path = '/home/school/masters/Scripts/coconut_full/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
new_iupac = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if row.err != 'no iupac':
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)
            if iupac:
                new_iupac.append([cid, iupac])                
                print([cid,iupac])
            else:
                print([cid,errorz])
        count += 1

new_iupac = pd.DataFrame(new_iupac,columns=['cid','iupac_name'])
df_old_iupac = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')

iupac_map = new_iupac.set_index('cid')['iupac_name']

df_old_iupac['iupac_name'] = df_old_iupac['iupac_name'].fillna(df_old_iupac['cid'].map(iupac_map))

df_old_iupac.to_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv',index= False)


[87443963, '(6E,10E,14E,16E,18E,20E,22E,26E)-31-methoxy-2,6,10,14,19,23,27,31-octamethyldotriaconta-6,10,14,16,18,20,22,26-octaen-2-ol']
[10754966, '(NE)-N-(12-pyridin-3-yldodecylidene)hydroxylamine']
[92033736, '(Z)-N-(2-methylpropyl)non-2-en-6,8-diynamide']
[164449732, '[2-[(Z)-hexadec-7-enoyl]oxy-3-[hydroxy-[2,3,4,5-tetrahydroxy-6-[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl]oxycyclohexyl]oxyphosphoryl]oxypropyl] (Z)-octadec-11-enoate']
[135601000, '(NZ)-N-[(16E)-16-hydroxyimino-13-methyl-3-prop-2-enoxy-6,9,11,12,14,15-hexahydrocyclopenta[a]phenanthren-17-ylidene]hydroxylamine']
[90478576, 'N-[(E)-3-(2-amino-1H-imidazol-5-yl)prop-2-enyl]-4,5-dibromo-1H-pyrrole-2-carboxamide;hydrochloride']
[5388464, '(4Z)-4-[(2-fluorophenyl)methylidene]-2-methyl-1,3-oxazol-5-one']
[89042540, '[(Z)-3-(4-methoxyphenyl)-4-phenylbut-2-en-2-yl] 4-methylbenzenesulfonate']
[44575966, '[(2S,3R,4S,5S)-5-hydroxy-2-[[(3S,8R,9S,10R,13S,14S,16S,17S)-17-hydroxy-10,13-dimethyl-17-[(2S)-6-methyl-3-oxoheptan-2-yl]-3

# 7 Sep

## now that I have all the iupacs that are listed by pubchem but were blank in the coconut
- going to do some stats to see what was added and what existed
- going to add the new iupacs

In [ ]:
import pandas as pd

df_old_less_iupac = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')

total_compounds = len(df_old_less_iupac)
number_of_iupac_missing = df_old_less_iupac['iupac_name'].isna().sum()
number_of_iupac = df_old_less_iupac['iupac_name'].notna().sum()

num_no_iupac_but_has_cid = (df_old_less_iupac['iupac_name'].isna() & df_old_less_iupac['cid'].notna()).sum()

df_new_iupacs = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')

new_iupacs = df_new_iupacs['iupac_name'].notna().sum()



out = f'Total coconut compounds: {total_compounds} \nNumber of iupac missinng from origional coconut download: {number_of_iupac_missing}\n Number of iupac in og: {number_of_iupac}\n Number of compounds in coconut og missing their iupac but had a cid: {num_no_iupac_but_has_cid}\n Number of new iupacs added from pubchem: {new_iupacs}'

print(out)

/tmp/ipykernel_12628/1765527528.py:3: DtypeWarning: Columns (0: np_classifier_is_glycoside) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old_less_iupac = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')


Total coconut compounds: 738827 
Number of iupac missinng from origional coconut download: 74593
 Number of iupac in og: 664234
 Number of compounds in coconut og missing their iupac but had a cid: 54988
 Number of new iupacs added from pubchem: 52469


## now adding the new iupacs to a new file getting ready for DB insertion (still need to try opsin)

In [2]:
og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')
new = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')


iupac_map = new.set_index('cid')['iupac_name']

og['iupac_name'] = og['iupac_name'].fillna(og['cid'].map(iupac_map))

og.to_csv('/home/school/masters/Scripts/coconut_full/coconut_with_pubchem_iupacs.csv', index=False)



/tmp/ipykernel_12628/2230518809.py:1: DtypeWarning: Columns (0: np_classifier_is_glycoside) have mixed types. Specify dtype option on import or set low_memory=False.
  og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

# the above code snippet cant work becauase there are cid duplicates
- trying to figure this out now

In [7]:
import pandas as pd
og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)
new = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')

og = og.dropna(subset='cid')
og[og.duplicated(subset='cid')]
num = og.duplicated(subset='cid').sum()

print('total dups:' + str(num) + '\n')
print(og['cid'])

total dups:14100

0          16400397.0
1            177785.0
2          10817089.0
3          39377583.0
4         145705321.0
             ...     
738821      6217354.0
738822      1778775.0
738823     16397128.0
738824      1590746.0
738826     16408156.0
Name: cid, Length: 636970, dtype: float64


## the plan now is to deduplicate the new iupacs file so i can add it to the others (I am assuming things with the same cid by definition have the same iupc name)

In [8]:
og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')
new = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')


new = new.dropna(subset='cid')
new = new.drop_duplicates(subset='cid', keep='first')
iupac_map = new.set_index('cid')['iupac_name']

og['iupac_name'] = og['iupac_name'].fillna(og['cid'].map(iupac_map))

og.to_csv('/home/school/masters/Scripts/coconut_full/coconut_with_pubchem_iupacs.csv', index=False)

/tmp/ipykernel_15073/901367274.py:1: DtypeWarning: Columns (0: np_classifier_is_glycoside) have mixed types. Specify dtype option on import or set low_memory=False.
  og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')


## Sorting out the deduplication issue 

In [ ]:
import pandas as pd

df = pd.read_csv('coconut_full/coconut_with_pubchem_iupacs.csv')

df = df[['identifier','standard_inchi','standard_inchi_key','name','iupac_name','annotation_level','np_likeness','collections','dois','synonyms','cas','coconut_id','cid']]

inchi_set = set()
new_df = []

counter = 0
for row in df.itertuples():
    inchi = None
    inchi_key = None
    name = None
    iupac = None
    annotation_level = None
    np_likeness = None
    collections = None
    dois = None
    synonyms = None
    cas = None
    coconut_id = None
    cid = None

    if row.standard_inchi not in inchi_set:
        inchi = row.standard_inchi
        inchi_key = row.standard_inchi_key
        inchi_set.add(row.standard_inchi)
        dupes = df.loc(df['standard_inchi']==inchi)

        for line in dupes.itertuples():
            if line.name not in name:
                name.append(line.name)
            if line.iupac_name not in iupac:
                iupac.append(line.iupac_name)
            if line.annotation_level not in annotation_level:
                annotation_level.append(line.annotation_level)
            if line.np_likeness not in np_likeness:
                np_likeness.append(line.np_likeness)
            if line.collections not in collections:
                collections.append(line.collections)
            if line.dois not in dois:
                dois.append(line.dois)
            if line.synonyms not in synonyms:
                synonyms.append(line.synonyms)
            if line.cas not in cas:
                cas.append(line.cas)
            if line.coconut_id not in coconut_id:
                coconut_id.append(line.coconut_id)
            if line.cid not in cid:
                cid.append(line.cid)
    name = name.to_list('|')
    iupac = iupac.to_list('|')
    annotation_level = annotation_level.to_list('|')
    np_likeness = np_likeness.to_list('|')
    collections = collections.to_list('|')
    dois = dois.to_list('|')
    synonyms = synonyms.to_list('|')
    cas = cas.to_list('|')
    coconut_id = coconut_id.to_list('|')
    cid = cid.to_list('|')

    new_df.append([row.identifier, inchi, inchi_key, name, iupac, annotation_level, np_likeness, collections, dois, synonyms, cas, coconut_id, cid])

df = pd.DataFrame(new_df, columns=['identifier','standard_inchi','standard_inchi_key','name','iupac_name','annotation_level','np_likeness','collections','dois','synonyms','cas','coconut_id','cid'])
df.to_csv('coconut_full/coconut_with_pubchem_iupacs_no_dupes.csv', index=False)



## I am keeping the above but It was never run. It is apparently very slow and gropby is supposed to be better so I am going to try that instead on my own and then let AI help me fix if it is dodgy

In [12]:
import pandas as pd

def neat_list(strrr):
    out = []
    for strr in strrr:
        if pd.notna(strr):
            out.append(str(strr).strip())
    return '|'.join(out)    

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_pubchem_iupacs.csv',low_memory=False)

df = df[['identifier','standard_inchi','standard_inchi_key','name','iupac_name','annotation_level','np_likeness','collections','dois','synonyms','cas','coconut_id','cid']]

df_new = df.groupby('standard_inchi',as_index=False,sort=False).agg({'name' : neat_list,
                                                                    'iupac_name': neat_list,
                                                                    'annotation_level': neat_list,
                                                                    'np_likeness': neat_list,
                                                                    'collections': neat_list,
                                                                    'dois': neat_list,
                                                                    'synonyms': neat_list,
                                                                    'cas': neat_list,
                                                                    'coconut_id': neat_list,
                                                                    'cid': neat_list})


df_new.to_csv('/home/school/masters/Scripts/coconut_full/coconut_iupacs_inchi_deduplicated.csv', index=False)